## Exercise Set 1

### Multiple Choice
**Q1.** Which of these are NOT key components for an optimization model?
- decision variables
- parameters
- objective function
- hyperparameters v

**Q2.** MO can be used as a strict replacement for nearly all ML problems
- True
- False v

**Q3.** ML can be used as a strict replacement for nearly all MO problems
- True
- False v

**Q4.** Which of the following is a type of decision variables in mathematical optimization?
- binary v
- complex
- feature
- complimentary

**Q5.** To solve a mixed-integer program, a solver may need to solve this many linear programs:
- none
- the number of decision variables
- the number of total constraints
- possibly exponentially many v

**Q6.** It is required that you or your data analytics team develop an expertise in `predictive` analytics before attempting `prescriptive` analytics (like mathematical optimization).
- True
- False v

**Q7.** The `feasible region` of a linear program (LP) will have ____ points in it than its corresponding mixed-integer program (MIP), assuming the two models are exactly the same other than the LP has only continuous variables and the MIP contains integer variables.
- more v
- less
- always exactly the same

**Q8.** Unless specified, the default variable type when using `addVars()` is **continuous**.
- True v
- False

Let $J = \{\texttt{Apple, Banana, Coconut, Dragonfruit, Elderberry, Fig, Gooseberry}\}$ and $T = \{1, 2, 3, 4\}$

**Q9-a.** Adding decision variables using `addVars(J,...)` and `addVars(range(8),...)` will add the *same* number of variables to a model
- True
- False v

**Q9-b.** Using the sets above, adding decision variables using `addVars(J, T,...)` and `addVars(range(28),...)` will add the *same* number of variables to a model
- True v
- False

### Formulation and Coding
Below is code for the entire original model in one cell if you would like to use it to help with these exercises.

In [1]:
%pip install gurobipy

import pandas as pd
import gurobipy as gp
from gurobipy import GRB

production = ['Baltimore','Cleveland','Little Rock','Birmingham','Charleston']
distribution = ['Columbia','Indianapolis','Lexington','Nashville','Richmond','St. Louis']

path = 'https://raw.githubusercontent.com/Gurobi/modeling-examples/master/optimization101/Modeling_Session_1/'
transp_cost = pd.read_csv(path + 'cost.csv', index_col=[0,1]).squeeze("columns")
max_prod = pd.Series([180,200,140,80,180], index = production, name = "max_production")
n_demand = pd.Series([89,95,121,101,116,181], index = distribution, name = "demand")
frac = 0.75

m = gp.Model('widgets')
x = m.addVars(production, distribution, name = 'prod_ship')
meet_demand = m.addConstrs((gp.quicksum(x[p,d] for p in production) >= n_demand[d] for d in distribution), name = 'meet_demand')
can_produce = m.addConstrs((gp.quicksum(x[p,d] for d in distribution) <= max_prod[p] for p in production), name = 'can_produce')
must_produce = m.addConstrs((gp.quicksum(x[p,d] for d in distribution) >= frac*max_prod[p] for p in production), name = 'must_produce')
m.setObjective(gp.quicksum(transp_cost[i,j]*x[i,j] for i in production for j in distribution), GRB.MINIMIZE)
m.optimize()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 100.0 MB/s  0:00:00
Restricted license - for non-production use only - expires 2028-11-29
Gurobi Optimizer version 14.0.0 build v14.0.0rc1 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 16 rows, 30 columns and 90 nonzeros (Min)
Model fingerprint: 0x20186c14
Model has 30 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 7e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e+01, 2e+02]

Presolve removed 5 rows and 0 columns
Presolve time: 0.02s
Presolved: 11 rows, 35 columns, 65 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.610000e+02   0.000000e+00      0s
      15    1.7048900e+03   0.000000e+00   0.000000e+00      0s

Solved in 15 iterations

You are told there is a new policy for transporting **widgets** from production facilities. It is now required that the minimum number of widgets shipped from any production facility to any distribution center needs to be at least 20.

**Q10-a.** Write out how the formulation changes using mathematical notation given the new requirement.

x[p,d] >= 20 for all p in production and for all d in distribution if x[p,d] != 0

**Q10-b.** Write the changes for **Q10-a** in gurobipy code in the cell below (no need to run it, unless you want to copy the model here to check).

In [25]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

production = ['Baltimore','Cleveland','Little Rock','Birmingham','Charleston']
distribution = ['Columbia','Indianapolis','Lexington','Nashville','Richmond','St. Louis']

path = 'https://raw.githubusercontent.com/Gurobi/modeling-examples/master/optimization101/Modeling_Session_1/'
transp_cost = pd.read_csv(path + 'cost.csv', index_col=[0,1]).squeeze("columns")
max_prod = pd.Series([180,200,140,80,180], index = production, name = "max_production")
n_demand = pd.Series([89,95,121,101,116,181], index = distribution, name = "demand")
frac = 0.75

m = gp.Model('widgets')
x = m.addVars(production, distribution, name = 'prod_ship')

minimum_shipments = m.addConstrs((x[p,d] >= 20 for p in production for d in distribution))
s = m.addVars(production, distribution, vtype=GRB.BINARY, name='on')
meet_demand = m.addConstrs((gp.quicksum(x[p,d]*s[p,d] for p in production) >= n_demand[d] for d in distribution), name = 'meet_demand')
can_produce = m.addConstrs((gp.quicksum(x[p,d]*s[p,d] for d in distribution) <= max_prod[p] for p in production), name = 'can_produce')
must_produce = m.addConstrs((gp.quicksum(x[p,d]*s[p,d] for d in distribution) >= frac*max_prod[p] for p in production), name = 'must_produce')
m.setObjective(gp.quicksum(transp_cost[i,j]*x[i,j]*s[i,j] for i in production for j in distribution), GRB.MINIMIZE)


In [26]:
m.optimize()

Gurobi Optimizer version 14.0.0 build v14.0.0rc1 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 30 rows, 60 columns and 30 nonzeros (Min)
Model fingerprint: 0x90ea4c9f
Model has 0 linear objective coefficients
Model has 30 quadratic objective terms
Model has 16 quadratic constraints
Variable types: 30 continuous, 30 integer (30 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [3e+00, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+01, 2e+01]
  QRHS range       [6e+01, 2e+02]

Presolve removed 30 rows and 0 columns
Presolve time: 0.01s
Presolved: 76 rows, 150 columns, 240 nonzeros
Presolved model has 60 SOS constraint(s)
Variable types: 90 continuous, 60 integer (60 binary)
Found heuristic solution: objective 2

In [13]:
x_sol = pd.Series(m.getAttr('x', x), index=transp_cost.index, name='quantity')
x_sol[x_sol > 0]

production   distribution
Baltimore    Nashville        19.0
             Richmond        116.0
Cleveland    Columbia         89.0
             Indianapolis     95.0
             Nashville         2.0
Little Rock  St. Louis       140.0
Birmingham   Nashville        80.0
Charleston   Lexington       121.0
             St. Louis        41.0
Name: quantity, dtype: float64

In [35]:
x_values = pd.Series(m.getAttr('x', x), index=transp_cost.index)
s_values = pd.Series(m.getAttr('X', s), index=transp_cost.index)
x_sol = x_values * s_values
x_sol = x_sol.rename('quantity')
x_sol[x_sol > 0]

production   distribution
Baltimore    Nashville        21.0
             Richmond        116.0
Cleveland    Columbia         89.0
             Indianapolis     95.0
Little Rock  St. Louis       140.0
Birmingham   Nashville        80.0
Charleston   Lexington       121.0
             St. Louis        41.0
Name: quantity, dtype: float64

In [33]:
s_sol = pd.Series(m.getAttr('X', s), index=transp_cost.index, name='switch')
s_sol.reset_index().pivot(index='production', columns='distribution', values='switch')

distribution,Columbia,Indianapolis,Lexington,Nashville,Richmond,St. Louis
production,,,,,,
Baltimore,0.0,0.0,0.0,1.0,1.0,0.0
Birmingham,0.0,0.0,0.0,1.0,0.0,0.0
Charleston,0.0,0.0,1.0,0.0,0.0,1.0
Cleveland,1.0,1.0,0.0,0.0,0.0,0.0
Little Rock,0.0,0.0,0.0,0.0,0.0,1.0


The initial widget model $m$ represented a single time period (e.g. a week, month, quarter). Suppose we added a time component using a set $T = \{0, 1, 2, 3\}$ representing a quarter of a year.

**Q11-a.** Use `addVar()` or `addVars()` to create a decision variable in gurobipy that represents the number of widgets shipped from a production facility to a distribution center for a given time period.

In [37]:
T = [0,1,2,3]
x = m.addVars(production, distribution, T, name = 'prod_ship')

**Q11-b.** To reference a time period before a given time $t$, we can use $t-1$ as a subscript (since $T$ is a set of integers), but $t-1$ doesn't work when $t = 0$ since $-1$ isn't in $T$. Fill in the ??? in the code below to represent a set of constraints that limits the amount a production facility can produce to `max_prod[p]` over *two consecutive* time periods.

In [38]:
time_prod_limit = m.addConstrs((gp.quicksum(x[p,d,t-1] + x[p,d,t] for d in distribution) <= max_prod[p] for p in production for t in T if t >0), name = 'time_prod_limit')